# 01 — Data Understanding and Initial Profiling

**Project:** AI Usage and Impact on Students and Professionals

## Learning goals
By the end of this notebook, you should be able to:
- Load a CSV file with Pandas.
- inspect its size, columns, and data types.
- distinguish missing values from duplicate records.
- identify categorical and numerical fields.
- produce an initial data-quality profile.

**Important:** This notebook does not clean or overwrite the original data. We are investigating first.

## 1. Import the libraries

Pandas helps us work with tabular data. NumPy provides numerical tools. We will import Matplotlib and Seaborn later when we begin visual analysis.

In [2]:
%pip install pandas numpy

  Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl.metadata (16 kB)
Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl (10.7 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 539.7 kB/s eta 0:00:23
   - -------------------------------------- 0.5/12.6 MB 539.7 kB/s eta 0:00:23
   -- ------------------------------------- 0.8/12.6 MB 555.1 kB/s eta 0:00:22
   -- ------------------------------------- 0.8/12.6 MB 555.1 kB/s eta


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 2. Load the original CSV

The file is stored in `data/raw/`. We preserve this original copy and will create a separate processed file after agreeing on cleaning decisions.

In [4]:
data_path = "../data/raw/AI_Usage_and_Impact_on_Students_and_Professionals.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3. Inspect the shape and preview

`shape` returns (number of rows, number of columns). `head()` displays the first five records.

In [5]:
print("Rows and columns:", df.shape)
display(df.head())

Rows and columns: (300, 19)


,User_ID,Age,Gender,Country,User_Type,Education_Level,Profession,Monthly_Income,AI_Tool,AI_Purpose,AI_Usage_Hours_Per_Day,Tasks_Performed,Monthly_AI_Cost,Productivity_Score,Accuracy_Rating,Satisfaction_Score,Time_Saved_Hours_Per_Week,Work_or_Study_Hours_Per_Day,Would_Recommend
0,U0269,30,Male,United Kingdom,Professional,Master's,Business Analyst,5421,ChatGPT,Office Work,2 hrs,"Data analysis using AI-powered analytics tools, automated report generation, process optimizatio...",NaN,5.3,4.0,7.0,3.9,7.3,Yes
1,U0217,30,Male,United States,Professional,Bachelor's,Customer Support Specialist,8903,ChatGPT,Customer Support,5.7,"Resolving customer inquiries using AI chatbot platforms, drafting personalized email responses w...",0,9.9,3.0,9.0,11.4,8.4 hrs,Yes
2,U0152,17,Male,Canada,Student,NaN,Computer Science Student,402,Chat GPT,Coding,1 hrs,NaN,20,3.2,2.0,4.0,5.2,6.7,No
3,U0264,35,FEMALE,Brazil,Professional,Bachelor's,Desenvolvedora de Software / Software Developer,1367,Meta AI,Coding,1.7,"Usar GitHub Copilot para autocompletar código, debugar aplicações com assistência de IA, otimiza...",20,5.6,4.0,5.0,6.4,9.6,Maybe
4,U0263,19,Female,Canada,Student,Undergraduate,Biology Student,438,Perplexity AI,NaN,4.1,"Using AI to generate study summaries for exam preparation, getting help with practice problem so...",0,4.7,4.0,7.0,9.2,6.2,Yes


## 4. Inspect column names and data types

A column's data type tells us how Pandas currently interprets its values. A numeric-looking column can still be stored as text if values contain units such as `hrs` or currency symbols.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   User_ID                      300 non-null    str    
 1   Age                          277 non-null    str    
 2   Gender                       290 non-null    str    
 3   Country                      290 non-null    str    
 4   User_Type                    300 non-null    str    
 5   Education_Level              287 non-null    str    
 6   Profession                   285 non-null    str    
 7   Monthly_Income               290 non-null    str    
 8   AI_Tool                      289 non-null    str    
 9   AI_Purpose                   287 non-null    str    
 10  AI_Usage_Hours_Per_Day       293 non-null    str    
 11  Tasks_Performed              282 non-null    str    
 12  Monthly_AI_Cost              288 non-null    str    
 13  Productivity_Score           28

In [7]:
display(pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "unique_non_null": df.nunique(dropna=True).values
}))

,column,data_type,missing_count,unique_non_null
0,User_ID,str,0,288
1,Age,str,23,52
2,Gender,str,10,11
3,Country,str,10,20
4,User_Type,str,0,2
5,Education_Level,str,13,16
6,Profession,str,15,126
7,Monthly_Income,str,10,233
8,AI_Tool,str,11,16
9,AI_Purpose,str,13,8


## 5. Check missing values

A missing value is not automatically an error. We need to understand how much is missing and whether it is concentrated in particular fields or groups.

In [8]:
missing_profile = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_percent=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_percent", ascending=False)
)

display(missing_profile)

,missing_count,missing_percent
Age,23,7.67
Tasks_Performed,18,6.00
Productivity_Score,16,5.33
Profession,15,5.00
Accuracy_Rating,15,5.00
AI_Purpose,13,4.33
Education_Level,13,4.33
Monthly_AI_Cost,12,4.00
AI_Tool,11,3.67
Time_Saved_Hours_Per_Week,10,3.33


## 6. Check duplicate rows and repeated user IDs

These are different checks:
- An exact duplicate row repeats every field.
- A repeated `User_ID` means the identifier appears more than once; it needs investigation, but it is not automatically safe to delete.

In [9]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated User_ID records:", df["User_ID"].duplicated(keep=False).sum())

repeated_ids = df[df["User_ID"].duplicated(keep=False)].sort_values("User_ID")
display(repeated_ids)

Exact duplicate rows: 12
Repeated User_ID records: 24


,User_ID,Age,Gender,Country,User_Type,Education_Level,Profession,Monthly_Income,AI_Tool,AI_Purpose,AI_Usage_Hours_Per_Day,Tasks_Performed,Monthly_AI_Cost,Productivity_Score,Accuracy_Rating,Satisfaction_Score,Time_Saved_Hours_Per_Week,Work_or_Study_Hours_Per_Day,Would_Recommend
235,U0055,30,Male,Philippines,Professional,Masters,Senior Software Engineer,579,Microsoft Copilot,Coding,3.8,"Code review and optimization using AI assistants, automated testing and debugging with AI tools,...",0,6.1,4.0,6.0,9.4 hrs,7.4,Maybe
85,U0055,30,Male,Philippines,Professional,Masters,Senior Software Engineer,579,Microsoft Copilot,Coding,3.8,"Code review and optimization using AI assistants, automated testing and debugging with AI tools,...",0,6.1,4.0,6.0,9.4 hrs,7.4,Maybe
16,U0059,22,Male,Canada,Professional,High School,Customer Service Representative,3546,Perplexity AI,Customer Support,1.7,"Responding to customer inquiries via AI chatbots, drafting personalized support emails using AI ...",0,7.0,4.0,7.0,3.4,6.7,Yes
87,U0059,22,Male,Canada,Professional,High School,Customer Service Representative,3546,Perplexity AI,Customer Support,1.7,"Responding to customer inquiries via AI chatbots, drafting personalized support emails using AI ...",0,7.0,4.0,7.0,3.4,6.7,Yes
251,U0085,17,Female,Canada,Student,PhD,Theoretical Physics Student,518,NaN,Studying,3.2,"Generating practice problems for quantum mechanics, Getting explanations for complex mathematica...",20,5.2,4.0,8.0,4.8,20,Yes
161,U0085,17,Female,Canada,Student,PhD,Theoretical Physics Student,518,NaN,Studying,3.2,"Generating practice problems for quantum mechanics, Getting explanations for complex mathematica...",20,5.2,4.0,8.0,4.8,20,Yes
159,U0104,26,Male,Pakistan,Student,NaN,Mechanical Engineering Student,54,ChatGPT,Research,2.7,"Literature review automation, research paper summarization, data analysis for experiments, thesi...",20,NaN,3.0,5.0,5 hrs,6.9,Maybe
271,U0104,26,Male,Pakistan,Student,NaN,Mechanical Engineering Student,54,ChatGPT,Research,2.7,"Literature review automation, research paper summarization, data analysis for experiments, thesi...",20,NaN,3.0,5.0,5 hrs,6.9,Maybe
67,U0116,20,Male,India,Student,Undergraduate,Digital Media and Communications Student,79,Google Gemini,Content Creation,3.5,"AI-assisted blog writing and SEO optimization, video scriptwriting and thumbnail design generati...",20,4.7,4.0,7.0,8.3,6.2,Yes
63,U0116,20,Male,India,Student,Undergraduate,Digital Media and Communications Student,79,Google Gemini,Content Creation,3.5,"AI-assisted blog writing and SEO optimization, video scriptwriting and thumbnail design generati...",20,4.7,4.0,7.0,8.3,6.2,Yes


## 7. Identify categorical and numeric-looking fields

The source description says the dataset contains demographic, education/employment, AI usage, and impact information. We will compare this description with the actual columns.

In [10]:
categorical_candidates = [
    "Gender", "Country", "User_Type", "Education_Level",
    "Profession", "AI_Tool", "AI_Purpose", "Would_Recommend"
]

numeric_candidates = [
    "Age", "Monthly_Income", "AI_Usage_Hours_Per_Day",
    "Monthly_AI_Cost", "Productivity_Score", "Accuracy_Rating",
    "Satisfaction_Score", "Time_Saved_Hours_Per_Week",
    "Work_or_Study_Hours_Per_Day"
]

print("Categorical candidates present:")
print([c for c in categorical_candidates if c in df.columns])

print("\nNumeric candidates present:")
print([c for c in numeric_candidates if c in df.columns])

Categorical candidates present:
['Gender', 'Country', 'User_Type', 'Education_Level', 'Profession', 'AI_Tool', 'AI_Purpose', 'Would_Recommend']

Numeric candidates present:
['Age', 'Monthly_Income', 'AI_Usage_Hours_Per_Day', 'Monthly_AI_Cost', 'Productivity_Score', 'Accuracy_Rating', 'Satisfaction_Score', 'Time_Saved_Hours_Per_Week', 'Work_or_Study_Hours_Per_Day']


## 8. Inspect category labels

Look for differences such as `Male` versus `male`, or `ChatGPT` versus `Chat GPT`. Do not standardise them until we agree on a mapping.

In [11]:
for column in categorical_candidates:
    if column in df.columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False).head(25))


Gender
Gender
Male          122
Female        105
male           12
Other          12
M              11
NaN            10
F              10
MALE            6
FEMALE          5
other           3
female          3
Non-binary      1
Name: count, dtype: int64

Country
Country
India                       42
United States               34
Canada                      32
Pakistan                    29
United Kingdom              26
Philippines                 24
Brazil                      23
Australia                   20
Nigeria                     17
Germany                     13
NaN                         10
France                      10
Kenya                        6
brazil                       4
india                        3
canada                       2
france                       1
U.K.                         1
United States of America     1
philippines                  1
australia                    1
Name: count, dtype: int64

User_Type
User_Type
Student         158
Professi

## 9. Compare the CSV with the supplied description

The description mentions `AI_Experience_Years`, but the CSV's 19 columns do not currently include that field. Record this as a documentation/data discrepancy; do not invent the missing column.

In [ ]:
description_claimed_column = "AI_Experience_Years"

print("Does the CSV contain AI_Experience_Years?",
      description_claimed_column in df.columns)
print("\nActual columns:")
print(df.columns.tolist())

## 10. Initial questions for the analyst

Answer these in your own words before moving to data cleaning:

1. What does one row appear to represent?
2. Why should we investigate repeated `User_ID` values before deleting records?
3. Why might `Monthly_Income` need a currency clarification?
4. Why is `AI_Experience_Years` a data documentation issue?
5. Why should we preserve the raw CSV?

## Phase 1 checkpoint

Save your observations in `reports/`. Do not modify the raw CSV. The next notebook will focus on validating data quality and agreeing on cleaning rules.